In [0]:
# =============================================================================
# Ground Truth Data Creation and Management
# =============================================================================
# This notebook creates and manages ground truth data for model validation.
# It handles data ingestion, deduplication, and time-based partitioning.
# =============================================================================

# Import required libraries
from pyspark.sql.functions import (
    col,
    date_format,
    lit,
    year,
    month,
    dayofmonth,
    hour,
    row_number,
    current_timestamp,
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
)
from datetime import datetime
import uuid

# =============================================================================
# Configuration and Setup
# =============================================================================
# Get configuration parameters from Databricks widgets
input_table_path = dbutils.widgets.get("input_table_path")
id_col = dbutils.widgets.get("ID_COL")
ground_truth_col = dbutils.widgets.get("GROUND_TRUTH_COL")
ground_truth_table = dbutils.widgets.get("GROUND_TRUTH_TABLE")
job_frequency = dbutils.widgets.get("JOB_FREQUENCY")
project_name = dbutils.widgets.get("PROJECT_NAME")

# Generate unique identifier for this batch
batch_uuid = uuid.uuid4().hex

print(f"🔧 Configuration:")
print(f"   Input Path: {input_table_path}")
print(f"   Ground Truth Table: {ground_truth_table}")
print(f"   Job Frequency: {job_frequency}")
print(f"   Batch UUID: {batch_uuid}")


# =============================================================================
# Helper Function: Delta Table Merge with Deduplication
# =============================================================================
def merge_with_delta_table(table_name, new_df, job_frequency, id_column="id"):
    """
    Merge new records into Delta table with duplicate prevention.

    This function:
    1. Creates time partitions based on job frequency
    2. Performs anti-join to find new records only
    3. Appends only new records to avoid duplicates

    Args:
        table_name: Target Delta table name
        new_df: DataFrame with new records to insert
        job_frequency: Frequency for time partitioning (hourly, daily, monthly, yearly)
        id_column: Column name for unique identification

    Returns:
        DataFrame with records that were actually inserted
    """
    from delta.tables import DeltaTable

    # Create time partition column based on frequency
    print(f"🕒 Creating {job_frequency} time partitions...")

    if job_frequency == "hourly":
        partition_format = "yyyy-MM-dd-HH"
    elif job_frequency == "daily":
        partition_format = "yyyy-MM-dd"
    elif job_frequency == "monthly":
        partition_format = "yyyy-MM"
    elif job_frequency == "yearly":
        partition_format = "yyyy"
    else:
        partition_format = "yyyy-MM-dd"  # Default to daily

    # Add time partition to new data
    new_df_with_partition = new_df.withColumn(
        "time_partition", date_format(col("timestamp"), partition_format)
    )

    # Read existing Delta table and add partition column
    delta_table = DeltaTable.forName(spark, table_name)
    existing_with_partition = delta_table.toDF().withColumn(
        "time_partition", date_format(col("timestamp"), partition_format)
    )

    # Find records that don't exist in the same time partition (anti-join)
    print(f"🔍 Identifying new records to insert...")
    records_to_insert = (
        new_df_with_partition.alias("new")
        .join(
            existing_with_partition.alias("existing"),
            (col(f"new.{id_column}") == col(f"existing.{id_column}"))
            & (col("new.time_partition") == col("existing.time_partition")),
            "left_anti",  # Only records that don't exist
        )
        .drop("time_partition")  # Remove temporary partition column
    )

    # Insert new records
    insert_count = records_to_insert.count()
    print(f"📝 Inserting {insert_count} new records...")

    if insert_count > 0:
        records_to_insert.write.format("delta").mode("append").option(
            "mergeSchema", "true"
        ).saveAsTable(table_name)
        print(f"✅ Successfully inserted {insert_count} records")
    else:
        print("ℹ️  No new records to insert")

    return records_to_insert


# =============================================================================
# Data Processing Pipeline
# =============================================================================

# Step 1: Load raw data
print(f"📚 Loading data from: {input_table_path}")
raw_data = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(input_table_path)
)

initial_count = raw_data.count()
print(f"📊 Loaded {initial_count} raw records")

# Step 2: Transform and prepare ground truth data
print(f"🔄 Transforming data for ground truth table...")
ground_truth_df = (
    raw_data.withColumn("uuid", lit(batch_uuid))  # Add batch identifier
    .withColumn("id", col(id_col).cast("string"))  # Ensure ID is string type
    .withColumnRenamed(ground_truth_col, "ground_truth")  # Standardize column name
    .withColumn(
        "ground_truth", col("ground_truth").cast("string")
    )  # Convert to string type
    .withColumn("project_name", lit(project_name))  # Add project identifier
    .withColumn("timestamp", current_timestamp())  # Add processing timestamp
    .select(
        "uuid", "id", "ground_truth", "project_name", "timestamp"
    )  # Select final columns
)

final_count = ground_truth_df.count()
print(f"📊 Prepared {final_count} records for ground truth table")

# Step 3: Write to Delta table (create or merge)
print(f"💾 Writing to ground truth table: {ground_truth_table}")

# Parse catalog, schema, and table from full table name
table_parts = ground_truth_table.split(".")
if len(table_parts) == 3:
    catalog_name, schema_name, table_name = table_parts

    # Set current catalog for Unity Catalog
    spark.sql(f"USE CATALOG {catalog_name}")
    print(f"✅ Using catalog: {catalog_name}")

    # Ensure catalog and schema exist
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
        print(f"✅ Catalog and schema verified: {catalog_name}.{schema_name}")
    except Exception as e:
        print(f"⚠️  Catalog/schema creation warning: {str(e)}")

# Display sample of data to be written
print(f"📋 Sample of ground truth data:")
ground_truth_df.show(5, truncate=False)

# Check if table exists with error handling
try:
    table_exists = spark.catalog.tableExists(ground_truth_table)
except Exception as e:
    print(f"⚠️  Could not check table existence: {str(e)}")
    table_exists = False

if table_exists:
    print("📋 Table exists - performing merge with deduplication...")
    result = merge_with_delta_table(
        ground_truth_table, ground_truth_df, job_frequency, "id"
    )

    # Show summary statistics
    result_count = result.count()
    print(f"📈 Processing Summary:")
    print(f"   - Input records: {initial_count}")
    print(f"   - Prepared records: {final_count}")
    print(f"   - New records inserted: {result_count}")

else:
    print("🆕 Table doesn't exist - creating new table...")
    print(f"💾 Writing {final_count} records in 'overwrite' mode...")
    ground_truth_df.write.format("delta").mode("overwrite").option(
        "mergeSchema", "true"
    ).option("overwriteSchema", "true").saveAsTable(ground_truth_table)

    print(f"📈 Processing Summary:")
    print(f"   - Input records: {initial_count}")
    print(f"   - Records written: {final_count}")
    print(f"✅ Ground truth table created successfully!")

# Verify write by reading back
result_count = spark.table(ground_truth_table).count()
print(f"📊 Verification: Table now contains {result_count} total records")

print(f"🏁 Ground truth processing completed for batch: {batch_uuid}")

# Optional: Display sample of the data for verification
print(f"📋 Sample of processed data:")
spark.table(ground_truth_table).orderBy(col("timestamp").desc()).limit(5).display()